# NL2SQL Agentic Loop

## Overview
A self-correcting ReAct agent that intercepts failed SQL predictions and actively retries them using chain-of-thought (CoT) reasoning and real-time execution error feedback.

---

## Ablation Study
The table below tracks the progressive accuracy gains across model sizes, prompt enhancements, fine-tuning, and agentic error correction:

| Step | Configuration / Change | Execution Accuracy |
| :--- | :--- | :--- |
| **Phi-2 2.7B Baseline** | Unaugmented schema | 40.00% |
| **Llama 3.1 8B** | Unaugmented schema | 70.89% |
| **+ Column Order Fix** | Normalize result tuple order | 72.00% |
| **+ Augmented Inference** | Sample rows injected (142 validation examples) | 75.00% |
| **+ Token Count Fix** | Scaled to all 301 wrong/error examples | 77.27% |
| **+ Augmented Fine-Tuning** | Trained on 6,607 examples | 81.00% |
| **+ Agentic Loop + CoT** | Chain-of-thought + execution retry on all failures | **84.00%** |

---

## Data Flow
When a query fails standard inference, it is routed through the agentic correction loop:

1. **Error Filtering (`validation_df`)** Isolates the 197 validation examples that resulted in wrong answers or execution errors.
2. **Contextual Prompting (`build_agentic_prompt()`)** Dynamically generates a detailed system instruction. Appends a CoT scaffold if the total prompt token count ($T$) is less than 1,400 tokens.
3. **Generation (`generate()`)** Generates text for a single example with `max_new_tokens=300` and a `repetition_penalty=1.2` to ensure distinct reasoning steps.
4. **Parsing (`extract_sql_from_reasoning()`)** Uses regex to extract the **last** ` ```sql ` block, falling back to the last instance of a `SELECT` statement if no code block is found.
5. **Execution Loop (`execute_sql_with_retry()`)** Runs the query against the database. If it fails, the error message is fed back into the prompt for a maximum of 3 correction attempts.
6. **Parallel Evaluation (`execution_accuracy_parallel()`)** Evaluates final corrected outputs across multiple threads.
7. **Final Performance Target** Elevates pipeline performance to an **84% execution accuracy**.

---

## Key Design Decisions

* **Dynamic CoT Guardrail ($T < 1400$):** The chain-of-thought scaffold (~100 tokens) is appended only if the prompt is short enough to leave sufficient budget for both reasoning steps and final SQL generation within the model's 2,048 token limit. Prompts exceeding this threshold skip CoT and generate raw SQL directly to prevent truncation.
* **Late-Block Extraction:** The extraction utility explicitly targets the *last* SQL block generated. Because the model frequently changes its mind and self-corrects mid-generation, capturing the final block ensures the most refined logic is tested.
* **Hybrid Parallelization Strategy:** Individual retry attempts for a single query depend directly on the sequential feedback of the previous error message and cannot be parallelized. However, distinct evaluation examples remain entirely independent, allowing them to be split concurrently across a `ThreadPoolExecutor`.
* **Database-in-the-Loop Feedback:** Upon an execution failure, the exact SQLite error string is injected back into the prompt context via `build_agentic_prompt(last_exec_sql, error_msg)`, giving the agent precise debugging signals.
* **Loop Prevention:** A strict repetition penalty of 1.2 is enforced during inference to prevent the model from getting stuck in repetitive phrase loops during longer CoT reasoning sequences.

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# PIP installs

In [2]:
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

!pip install --upgrade huggingface_hub -q
!pip install "torchvision>=0.27.0" --upgrade -q
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" --force-reinstall --no-cache-dir -q
!pip install --no-deps "trl<0.13.0" peft accelerate -q



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 77.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 254.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 134.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 235.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 157.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 291.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 315.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 242.0 MB/s eta 0:00:00
  

In [3]:
!pip install bitsandbytes --upgrade -q
import ctypes
ctypes.CDLL('/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13')
import bitsandbytes

In [4]:
# !pip install --upgrade huggingface_hub -q
# !pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" -q
# !pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" --force-reinstall --no-cache-dir -q
# !pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes -q

In [5]:
! pip show unsloth

Name: unsloth
Version: 2026.6.1
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: nest-asyncio, pydantic, pyyaml, typer
Required-by: 


# Loading Model

In [6]:
import os
from huggingface_hub import snapshot_download

# This is a safe, visible directory
local_model_path = "/kaggle/working/llama-3-1-clean"

snapshot_download(
    repo_id="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    local_dir=local_model_path,
    local_dir_use_symlinks=False, # FORCES real files, no broken pointers
    resume_download=True
)

from unsloth import FastLanguageModel
import torch
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "/kaggle/working/llama-3-1-clean", # No more cache!
    max_seq_length = 2048,
    dtype        = None,
    load_in_4bit = True,
    device_map = {"":0} # multi-GPU sharding is making your inference slower. forcing everything on cuda:0
) 

# Apply LoRA adapters

adapter_path = '/kaggle/input/notebooks/mehulkumar99/spider-1-question-to-sql-query-81-exec-accu/outputs/checkpoint-450/'
model = PeftModel.from_pretrained(model, adapter_path)

print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB")
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0608 10:13:00.155000 381 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0608 10:13:00.192000 381 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /kaggle/working/llama-3-1-clean as a legacy tokenizer.


Memory used: 5.76 GB
trainable params: 0 || all params: 8,072,204,288 || trainable%: 0.0000


# To create error message

In [7]:
import sqlite3

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
def execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR ):
    
    """Execute SQL against SQLite DB, return result set or None on error."""
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    # CRITICAL: Check if the file actually exists before connecting
    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)  # 'mode=ro' tells SQLite to open it in Read-Only mode
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result

    except sqlite3.Error as e:
        # This will return the specific SQLite error message (e.g., "no such column: name")
        print(f"SQL Error on {db_id}: {e}")
        return str(e)
    except Exception as e:
        # This catches other issues (like file path errors)
        print(f"General Error: {e}")
        return None

## Loading schema_lookup

In [8]:
import pickle

# Open the file in read-binary mode
with open('/kaggle/input/datasets/mehulkumar99/augmented-schema/schema_lookup_augmented.pkl', 'rb') as file:
    schema_lookup = pickle.load(file)

# View the data
print(schema_lookup['perpetrator'].keys())


dict_keys(['db_id', 'Schema (values (type))', 'Primary Keys', 'Foreign Keys', 'format_schema', 'augmented_schema'])


In [45]:
validation_df = pd.read_csv('/kaggle/input/datasets/mehulkumar99/81validation-df/81newest_validation_predictions (1).csv')
validation_df.head(3)

,db_id,query,question,schema,count_tables,predicted_sql,results,T,new_pred_sql
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM singer,correct,551,SELECT count(*) FROM singer
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM singer,correct,552,SELECT count(*) FROM singer
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","Show name, country, age for all singers ordere...","stadium : Stadium_ID (number) , Location (text...",4,"SELECT name, country, age FROM singer ORDER ...",correct,563,"SELECT name, country, age FROM singer ORDER ..."


# Loading 81% validation_df

In [48]:
#schema_lookup, index, dataset, last_exec_sql = None, error_msg = None, inference = False 
total = len(validation_df)
lengths = []
for i in range(total):
    prompt = build_agentic_prompt( schema_lookup, index = i , dataset = validation_df, inference = True )['text']
    lengths.append(len(tokenizer(prompt)['input_ids']))

validation_df['T'] = lengths

In [49]:
# Isolate the problematic rows
error_rows =  validation_df[validation_df['results'] == 'error'].copy()
wrong_rows =  validation_df[validation_df['results'] == 'wrong'].copy()
# err_wro_rows = pd.concat([error_rows, wrong_rows], ignore_index = True)
err_wro_rows = validation_df[validation_df['results'].isin(['wrong', 'error'])].copy()


print(f"Total Errors: {len(error_rows)}")
print(f"Total Wrong: {len(wrong_rows)}")
err_wro_rows.head(3)

Total Errors: 36
Total Wrong: 161


,db_id,query,question,schema,count_tables,predicted_sql,results,T,new_pred_sql
16,concert_singer,"select max(capacity), average from stadium",What is the maximum capacity and the average o...,"stadium : Stadium_ID (number) , Location (text...",4,"SELECT max(capacity), avg(Average) FROM stadium",wrong,724,"SELECT max(capacity), avg(Average) FROM stadium"
43,concert_singer,select count(*) from concert where stadium_id ...,Find the number of concerts happened in the st...,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM concert AS T1 JOIN stadiu...,wrong,726,SELECT count(*) FROM concert AS t1 JOIN stadiu...
44,concert_singer,select count(*) from concert where stadium_id ...,What are the number of concerts that occurred ...,"stadium : Stadium_ID (number) , Location (text...",4,SELECT count(*) FROM concert AS T1 JOIN stadiu...,wrong,728,SELECT count(*) FROM concert AS t1 JOIN stadiu...


In [50]:
max_t_df = validation_df.groupby('db_id')['T'].max().reset_index()
print(max_t_df)

                           db_id     T
0                   battle_death   699
1                          car_1   808
2                 concert_singer   737
3                   course_teach   490
4           cre_Doc_Template_Mgt   693
5                    dog_kennels  1696
6       employee_hire_evaluation   679
7                       flight_2   552
8                   museum_visit   538
9                      network_1   440
10                     orchestra   810
11                        pets_1   565
12                  poker_player   534
13        real_estate_properties  1253
14                        singer   508
15  student_transcripts_tracking  1695
16                        tvshow   897
17                       voter_1   506
18                       world_1   864
19                         wta_1  1248


In [51]:
wrong_rows['T'].max()

1691

In [52]:
def get_error_message(row):
    # We pass the predicted SQL and the db_id from the row
    return execute_sql(row['new_pred_sql'], row['db_id'])

# 3. Apply the function and create a new 'error_log' column
error_rows['error_log'] = error_rows.apply(get_error_message, axis=1)

SQL Error on pets_1: no such column: T
SQL Error on car_1: no such column: contid
SQL Error on car_1: no such column: Model
SQL Error on car_1: no such column: T2.Year
SQL Error on car_1: no such column: Make
SQL Error on car_1: no such column: T1.Maker
SQL Error on car_1: no such column: MPG
SQL Error on car_1: no such column: T1.Weight
SQL Error on car_1: no such column: CountryId
SQL Error on car_1: unrecognized token: "'"
SQL Error on car_1: no such column: T2.Weight
SQL Error on car_1: no such column: countryid
SQL Error on flight_2: no such column: T2.Country
SQL Error on cre_Doc_Template_Mgt: ambiguous column name: document_id
SQL Error on wta_1: Could not decode to UTF-8 column 'last_name' with text 'Treyes Albarrac��N'
SQL Error on wta_1: Could not decode to UTF-8 column 'last_name' with text 'Treyes Albarrac��N'
SQL Error on wta_1: no such column: T2.rank
SQL Error on student_transcripts_tracking: ambiguous column name: semester_id
SQL Error on student_transcripts_tracking: n

In [53]:
index = 4
db_id = error_rows['db_id'].iloc[index]
print(f'db_id    : {error_rows['db_id'].iloc[index]}')
print(f'question : {error_rows['question'].iloc[index]}')
print(f'Gold SQl : {error_rows['query'].iloc[index]}')
print(f'Pred_sql : {error_rows['new_pred_sql'].iloc[index]}')
print(f'wrror_msg: {error_rows['error_log'].iloc[index]}')
print(f'schema   : {schema_lookup[db_id]['augmented_schema']}')

db_id    : car_1
question : Find the make and production time of the cars that were produced in the earliest year?
Gold SQl : SELECT T2.Make ,  T1.Year FROM CARS_DATA AS T1 JOIN CAR_NAMES AS T2 ON T1.Id  =  T2.MakeId WHERE T1.Year  =  (SELECT min(YEAR) FROM CARS_DATA);
Pred_sql : SELECT Make,  YEAR FROM cars_data ORDER BY YEAR ASC LIMIT 1
wrror_msg: no such column: Make
schema   : # Table: continents 
## Columns:
- ContId (number, PK) | e.g: [1, 2, 3]
- Continent (text) | e.g: ['america', 'europe', 'asia']

# Table: countries 
## Columns:
- CountryId (number, PK) | e.g: [1, 2, 3]
- CountryName (text) | e.g: ['usa', 'germany', 'france']
- Continent (number, FK -> continents.ContId) | e.g: [1, 2, 2]

# Table: car_makers 
## Columns:
- Id (number, PK) | e.g: [1, 2, 3]
- Maker (text) | e.g: ['amc', 'volkswagen', 'bmw']
- FullName (text) | e.g: ['American Motor Company', 'Volkswagen', 'BMW']
- Country (text, FK -> countries.CountryId) | e.g: ['1', '2', '2']

# Table: model_list 
## Columns:

# Understanding wrong results

In [ ]:
wrong_rows.head(5)

## Filtering out greater 1400 tokens to make space for generation, chain of thought, etc

In [54]:
index = 77
db_id = wrong_rows['db_id'].iloc[index]
print(f'db_id    : {wrong_rows['db_id'].iloc[index]}')
print(f'question : {wrong_rows['question'].iloc[index]}')
print(f'Gold SQl : {wrong_rows['query'].iloc[index]}')
print(f'Pred_sql : {wrong_rows['new_pred_sql'].iloc[index]}')
# print(f'wrror_msg: {wrong_rows['error_log'].iloc[index]}')
print(f'schema   : {schema_lookup[db_id]['augmented_schema']}')

db_id    : student_transcripts_tracking
question : What are the first, middle, and last names, along with the ids, of all students who enrolled in 2 degree programs in one semester?
Gold SQl : SELECT T1.first_name ,  T1.middle_name ,  T1.last_name ,  T1.student_id FROM Students AS T1 JOIN Student_Enrolment AS T2 ON T1.student_id  =  T2.student_id GROUP BY T1.student_id HAVING count(*)  =  2
Pred_sql : SELECT T1.first_name,  T1.middle_name,  T1.last_name,  T1.student_id FROM Students AS T1 JOIN Student_Enrolment AS T2 ON T1.student_id  =  T2.student_id GROUP BY T2.degree_program_id HAVING count(*)  =  2
schema   : # Table: Addresses 
## Columns:
- address_id (number, PK) | e.g: [1, 2, 3]
- line_1 (text) | e.g: ['2294 Grant Square Apt. 235', '3999 Aufderhar Ways Suite 593', '67942 Carlotta Ferry Apt. 686']
- line_2 (text) | e.g: ['Apt. 370', 'Apt. 388', 'Apt. 583']
- line_3 (text) | e.g: [None, None, None]
- city (text) | e.g: ['Port Chelsea', 'Lake Laishafurt', 'Goodwinhaven']
- zip_pos

In [ ]:
# error_rows = error_rows [error_rows ['T'] <= 1400]
# wrong_rows = wrong_rows [wrong_rows ['T'] <= 1400]

# React agents

In [ ]:
# import sqlite3

# KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
# def execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR ):
    
#     """Execute SQL against SQLite DB, return result set or None on error."""
#     db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

#     # CRITICAL: Check if the file actually exists before connecting
#     if not os.path.exists(db_path):
#         print(f"Warning: Database file not found at {db_path}")
#         return None
#     try:
#         # 'mode=ro' tells SQLite to open it in Read-Only mode
#         conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
#         cursor = conn.cursor()
#         cursor.execute(sql)
#         result = cursor.fetchall()
#         conn.close()
#         return result
#     # except Exception as e:
#     #     return None   # invalid SQL
#     except sqlite3.Error as e:
#         # This will return the specific SQLite error message (e.g., "no such column: name")
#         print(f"SQL Error on {db_id}: {e}")
#         return str(e)
#     except Exception as e:
#         # This catches other issues (like file path errors)
#         print(f"General Error: {e}")
#         return None
    
# sql = 'SELECT T1.Model FROM model_list AS T1 JOIN car_names AS T2 ON T1.Model  =  T2.Model WHERE T2.MakeId  IN  (SELECT T2.MakeId FROM cars_data AS T3 JOIN car_names AS T2 ON T3.Id  =  T2.MakeId WHERE T3.Weight  <  (SELECT avg(T3.Weight) FROM cars_data AS T3))'
# sql1 = 'SELECT T1.model FROM CAR_NAMES AS T1 JOIN CARS_DATA AS T2 ON T1.MakeId  =  T2.Id WHERE T2.Weight  <  (SELECT avg(Weight) FROM CARS_DATA)'
# result = execute_sql(sql, db_id = 'car_1')
# result1 = execute_sql(sql1, db_id = 'car_1')
# print(len(result))
# print(len(result1))

In [55]:
def build_agentic_prompt(schema_lookup, index, dataset, last_exec_sql=None, error_msg=None, inference=True):

    df = dataset
    prompt = ''

    db_id = df['db_id'][index]
    question = df['question'][index]
    schema = schema_lookup[db_id]['augmented_schema']

    token_length = df['T'][index]
    # print(token_length)
    use_reasoning = token_length < 1400

    
    # Base system instructions that force the Chain-of-Thought
    system_instruction = (
        "You are an expert SQL Architect.\n"
        "STRICT RULE for STEP 2 (VERIFICATION):\n"
        "1. You must cross-reference the Question against Table Names and Sample Rows.\n"
        "2. DIRECTNESS: If multiple tables have the column (e.g., 'model'), pick the one that is the 'Primary' source (e.g., CAR_NAMES over MODEL_LIST).\n"
        "3. SAMPLES: If the question asks for 'names' but the sample rows for a column are IDs (1, 2, 3), you MUST JOIN to the parent table to get the text names.\n"
        "4. NO HALLUCINATION: Only use tables and columns present in the schema."
    )

    if not last_exec_sql and not error_msg:
        task_intro = "Analyze the question, follow system instructions , and generate the SQL."
    else:
        # The 'Correction' variant for the Agentic loop
        task_intro = (
            f"Fix the following broken SQL, given the error message when execution, and follow system instructions.\n"            
        )

    prompt = f"""<|start_header_id|>system<|end_header_id|>

{system_instruction}

{task_intro}<|eot_id|><|start_header_id|>user<|end_header_id|>

### Schema:
{schema}

### Question:
{question}"""

    if last_exec_sql and error_msg:
        prompt += f"""

### Failed SQL: 
{last_exec_sql}

### Execution Error: 
{error_msg}"""

    prompt += "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    

    # Force the model to output a reasoning block before the SQL
    if use_reasoning:
        text = prompt + (
            "### LOGIC ANALYSIS\n"
            "[STEP 1: ATTRIBUTES] The required data points are: \n"
            "[STEP 2: VERIFICATION] Table selection and sample check:\n"
            "1. Primary Column: " # Use numbering instead of just a dash to suggest a sequence
        )
    else:
        text = prompt
    return {'text': text}

index = 98
text = build_agentic_prompt( schema_lookup, index = index , dataset = validation_df, inference = True )
prompt = text['text']
print(f'prompt is: {text['text']}')
print(f'length of the prompt is: {len(tokenizer(prompt)['input_ids'])}')

prompt is: <|start_header_id|>system<|end_header_id|>

You are an expert SQL Architect.
STRICT RULE for STEP 2 (VERIFICATION):
1. You must cross-reference the Question against Table Names and Sample Rows.
2. DIRECTNESS: If multiple tables have the column (e.g., 'model'), pick the one that is the 'Primary' source (e.g., CAR_NAMES over MODEL_LIST).
3. SAMPLES: If the question asks for 'names' but the sample rows for a column are IDs (1, 2, 3), you MUST JOIN to the parent table to get the text names.
4. NO HALLUCINATION: Only use tables and columns present in the schema.

Analyze the question, follow system instructions , and generate the SQL.<|eot_id|><|start_header_id|>user<|end_header_id|>

### Schema:
# Table: continents 
## Columns:
- ContId (number, PK) | e.g: [1, 2, 3]
- Continent (text) | e.g: ['america', 'europe', 'asia']

# Table: countries 
## Columns:
- CountryId (number, PK) | e.g: [1, 2, 3]
- CountryName (text) | e.g: ['usa', 'germany', 'france']
- Continent (number, FK -> c

In [56]:
# def build_agentic_prompt(schema_lookup, index, dataset, last_exec_sql=None, error_msg=None, inference=True):

#     df = dataset
#     prompt = ''

#     db_id = df['db_id'][index]
#     question = df['question'][index]
#     schema = schema_lookup[db_id]['augmented_schema']

#     token_length = df['T'][index]
#     # print(token_length)
#     use_reasoning = token_length < 1400

    
#     # Base system instructions that force the Chain-of-Thought
#     system_instruction = (
#         "You are an expert SQL Architect.\n"
#         "STRICT RULES:\n"
#         "1. SAMPLES: Match string values EXACTLY as shown in sample rows including case.\n"
#         "2. DIRECTNESS: Pick the Primary source table for ambiguous columns.\n"
#         "3. SAMPLES: If question asks for names but sample rows show IDs, JOIN to parent table.\n"
#         "4. NO HALLUCINATION: Only use tables and columns present in the schema.\n"
#         "5. SET OPERATIONS: 'Both A and B' = INTERSECT. 'Either A or B' = IN or UNION.\n"
#         "6. AGGREGATION: Filter on aggregated values with HAVING not WHERE.\n"
#         "7. RANKINGS: Rank 1 is best. 'Highest rank' = lowest rank number = MIN.\n"
#         "8. SELECT: Return exactly what the question asks — no more, no less.\n"
#     )

#     if not last_exec_sql and not error_msg:
#         task_intro = "Analyze the question, follow system instructions , and generate the SQL."
#     else:
#         # The 'Correction' variant for the Agentic loop
#         task_intro = (
#             f"Fix the following broken SQL, given the error message when execution, and follow system instructions.\n"            
#         )

#     prompt = f"""<|start_header_id|>system<|end_header_id|>

# {system_instruction}

# {task_intro}<|eot_id|><|start_header_id|>user<|end_header_id|>

# ### Schema:
# {schema}

# ### Question:
# {question}"""

#     if last_exec_sql and error_msg:
#         prompt += f"""

# ### Failed SQL: 
# {last_exec_sql}

# ### Execution Error: 
# {error_msg}"""

#     prompt += "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    

#     # Force the model to output a reasoning block before the SQL
#     if use_reasoning:
#         text = prompt + (
#             "### LOGIC ANALYSIS\n"
#             "[STEP 1: ATTRIBUTES] The question asks for: \n"
#             "[STEP 2: TABLES] Tables needed and their relationships:\n"
#             "[STEP 3: CARDINALITY] For each JOIN, is it 1-to-many or many-to-many? Need DISTINCT or GROUP BY?\n"
#             "[STEP 4: SET LOGIC] Does the question require ALL conditions (INTERSECT) or ANY condition (IN/UNION)?\n"
#             "[STEP 5: AGGREGATION] Is filtering on an aggregated value? Use HAVING not WHERE.\n"
#             "[STEP 6: VERIFICATION] Does my SELECT return exactly what was asked?\n"
#             "[STEP 7: SQL]\n"
#         )
#     else:
#         text = prompt
#     return {'text': text}

# index = 744
# # index = 439
# # index = 526
# text = build_agentic_prompt( schema_lookup, index = index , dataset = validation_df, inference = True )
# prompt = text['text']
# print(f'prompt is: {text['text']}')
# print(f'length of the prompt is: {len(tokenizer(prompt)['input_ids'])}')

# Generating above prompt

In [57]:
def generate(prompt):
    tokenized = tokenizer(
    prompt,
    truncation=True,
    max_length=2048,
    padding=False,          # don't pad yet — DataLoader will handle per batch
    add_special_tokens= True,
    return_tensors='pt', )   # return list, not tensors


    input_ids = tokenized['input_ids'].to(model.device)
    # print(tokenizer.decode(input_ids[0]))
    attention_mask = tokenized['attention_mask'].to(model.device) 
    
    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=300,
        do_sample=False,
        use_cache=True,
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
    )

    generated = output[0][len(tokenized['input_ids'][0]):]
    sql = tokenizer.decode(generated, skip_special_tokens= True).strip()
    # if sql.startswith("assistant"):
    #     sql = sql[len("assistant"):].strip()

    return sql

sql = generate(prompt)
print(f'generated_sql is : {sql}')

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


generated_sql is : `Model` from `car_names`
2. Joining to `cars_data`: Since we need "Weight" which is not available directly under `car_names`, join it with `cars_data`

```sql
SELECT T1.model FROM car_names AS T1 JOIN cars_data AS T2 ON T1.makeid = T2.id WHERE T2.weight < (
    SELECT avg(Weight)
    FROM cars_data);
```

This query will return all models of cars whose weights are less than the overall average weight across all records.


In [58]:
sql

'`Model` from `car_names`\n2. Joining to `cars_data`: Since we need "Weight" which is not available directly under `car_names`, join it with `cars_data`\n\n```sql\nSELECT T1.model FROM car_names AS T1 JOIN cars_data AS T2 ON T1.makeid = T2.id WHERE T2.weight < (\n    SELECT avg(Weight)\n    FROM cars_data);\n```\n\nThis query will return all models of cars whose weights are less than the overall average weight across all records.'

In [59]:
import sqlite3

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
def execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR ):
    
    """Execute SQL against SQLite DB, return result set or None on error."""
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    # CRITICAL: Check if the file actually exists before connecting
    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None
    try:
        # 'mode=ro' tells SQLite to open it in Read-Only mode
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    # except Exception as e:
    #     return None   # invalid SQL
    except sqlite3.Error as e:
        # This will return the specific SQLite error message (e.g., "no such column: name")
        print(f"SQL Error on {db_id}: {e}")
        return str(e)
    except Exception as e:
        # This catches other issues (like file path errors)
        print(f"General Error: {e}")
        return None
    
gold_sql = 'SELECT T1.course_name ,  T1.course_id FROM Courses AS T1 JOIN Sections AS T2 ON T1.course_id  =  T2.course_id GROUP BY T1.course_id HAVING count(*)  <=  2'

sql1 = """SELECT T1.course_name,  T1.course_id FROM COURSES AS T1 JOIN SECTIONS AS T2 ON T1.course_id  =  T2.course_id GROUP BY T1.course_id HAVING count(*) <  2"""

# sql2 = """"""

wrong_sql = """SELECT T1.course_name FROM courses AS T1 JOIN sections AS T2 ON T1.course_id = T2.course_id GROUP BY T1.course_id HAVING count(*) < 2"""

# no_cot_sql = """SELECT count(*) FROM countrylanguage AS t1 JOIN countrylanguage AS t2 ON t1.countrycode  =  t2.countrycode AND t1.language!= t2.language GROUP BY t1.countrycode HAVING sum(CASE WHEN language  =  "english" THEN percentage ELSE NULL END) > 0 AND sum(CASE WHEN language  =  "dutch" THEN percentage ELSE NULL END) > 0"""


result = execute_sql(gold_sql,   db_id = 'student_transcripts_tracking')
result1 = execute_sql(sql1,      db_id = 'student_transcripts_tracking')
# result2 = execute_sql(sql2,      db_id = 'world_1')
result3 = execute_sql(wrong_sql, db_id = 'student_transcripts_tracking')
# result4 = execute_sql(no_cot_sql, db_id = 'world_1')

print(result)
print(result1)
# print(result2)
print(result3)
# print(result4)

[('ds', 1), ('math', 2), ('en', 4), ('fr', 5), ('la', 6), ('cal', 7), ('nlp', 8), ('dl', 9), ('ml', 10), ('db', 12), ('pl', 14)]
[('math', 2), ('en', 4), ('la', 6), ('cal', 7), ('dl', 9), ('ml', 10), ('db', 12)]
[('math',), ('en',), ('la',), ('cal',), ('dl',), ('ml',), ('db',)]


In [34]:
# print(validation_df['new_pred_sql'].iloc[index])
print(validation_df['query'].iloc[index])

SELECT COUNT(*) FROM (SELECT T1.Name FROM country AS T1 JOIN countrylanguage AS T2 ON T1.Code  =  T2.CountryCode WHERE T2.Language  =  "English" INTERSECT SELECT T1.Name FROM country AS T1 JOIN countrylanguage AS T2 ON T1.Code  =  T2.CountryCode WHERE T2.Language  =  "Dutch")


# sql --> Execution --> error --> New Prompt --> Generate --> Execution -->

In [60]:
import re
def extract_sql_from_reasoning(text):
    # 1. Try to find ALL triple backtick SQL blocks
    sql_blocks = re.findall(r"```sql\s+(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    
    if sql_blocks:
        # We take the LAST block, as models often 'correct' themselves in later blocks
        sql = sql_blocks[-1].strip()
    else:
        # 2. Fallback: If no backticks, find the LAST occurrence of 'SELECT'
        # This prevents grabbing reasoning text that happens to contain the word 'SELECT'
        upper_text = text.upper()
        if "SELECT" in upper_text:
            last_select_idx = upper_text.rfind("SELECT")
            sql = text[last_select_idx:].strip()
        else:
            sql = text.strip()


    # 3. Final Cleanup: Remove trailing text like "Note:" or "Therefore..."
    # Often models append conversational text after the code
    sql = sql.split("Note:")[0].split("Therefore:")[0].split("###")[0].strip()
    
    # 4. Remove comments to keep the 'new_pred_sql' column clean
    clean_lines = [line for line in sql.split('\n') if not line.strip().startswith('--')]
    return "\n".join(clean_lines).strip()

# text = sql
text = sql
sql = extract_sql_from_reasoning(text)
sql

'SELECT T1.model FROM car_names AS T1 JOIN cars_data AS T2 ON T1.makeid = T2.id WHERE T2.weight < (\n    SELECT avg(Weight)\n    FROM cars_data);'

In [61]:
import sqlite3
import os

def execute_sql_with_retry(max_try, index, db_dir, dataset):
    db_id = dataset['db_id'].iloc[index]
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None, ""

    # 1. CLEAN INITIAL CALL: Let the prompt function decide on the anchor
    # build_agentic_prompt already appends "Attributes needed: The" if T < 1400
    prompt_data = build_agentic_prompt(schema_lookup, index, dataset, inference=True)
    agentic_input = prompt_data['text']
    
    full_response = generate(agentic_input)
    sql = extract_sql_from_reasoning(full_response)

    # print(sql)

    for i in range(max_try):
        conn = None
        try:
            # Connect using read-only mode for safety
            conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
            cursor = conn.cursor()
            cursor.execute(sql)
            result = cursor.fetchall()
            if not result:
                error_message = "The query executed successfully but returned 0 results. Check if the JOIN columns (Foreign Keys) actually contain matching values."
                raise ValueError(error_message)
                
            # conn.close()
            return result, sql

        except (sqlite3.Error, Exception, ValueError) as e:
            # 2. This catches sqlite3.Error, ValueError, and everything else
            print(f"Attempt {i+1} failed on {db_id}: {e}")
            error_message = str(e)
            
            # 2. CLEAN RETRY CALL: Do not manually append the anchor string here
            # Passing last_exec_sql and error_msg allows the model to self-correct
            retry_prompt_data = build_agentic_prompt(
                schema_lookup, 
                index, 
                dataset, 
                last_exec_sql=sql, 
                error_msg=error_message, 
                inference=True
            )
            
            
            retry_input = retry_prompt_data['text']
            
            full_response = generate(retry_input)
            sql = extract_sql_from_reasoning(full_response)

        finally:
        # 4. This ALWAYS runs, ensuring your database doesn't leak connections
            if conn:
                conn.close()

    return None, sql


result, sql = execute_sql_with_retry(max_try = 3, index = index, db_dir =KAGGLE_DB_DIR, dataset = validation_df)
print(f'results is : {result}')
print(f'sql is : {sql}')

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


results is : [('toyota',), ('plymouth',), ('amc',), ('ford',), ('datsun',), ('volkswagen',), ('peugeot',), ('audi',), ('saab',), ('bmw',), ('amc',), ('datsun',), ('chevrolet',), ('toyota',), ('ford',), ('volkswagen',), ('amc',), ('amc',), ('chevrolet',), ('mercury',), ('opel',), ('peugeot',), ('fiat',), ('toyota',), ('datsun',), ('volkswagen',), ('plymouth',), ('toyota',), ('dodge',), ('volkswagen',), ('chevrolet',), ('ford',), ('mazda',), ('volvo',), ('volkswagen',), ('peugeot',), ('renault',), ('ford',), ('datsun',), ('toyota',), ('dodge',), ('toyota',), ('amc',), ('plymouth',), ('volkswagen',), ('amc',), ('toyota',), ('chevrolet',), ('datsun',), ('mazda',), ('ford',), ('mercury',), ('fiat',), ('fiat',), ('opel',), ('audi',), ('volvo',), ('saab',), ('toyota',), ('ford',), ('amc',), ('datsun',), ('ford',), ('toyota',), ('chevrolet',), ('audi',), ('volkswagen',), ('opel',), ('toyota',), ('datsun',), ('dodge',), ('fiat',), ('fiat',), ('honda',), ('subaru',), ('fiat',), ('toyota',), ('fo

In [37]:
len(err_wro_rows.index.tolist())

197

In [62]:
error_indices = err_wro_rows.index.tolist()
from tqdm.auto import tqdm

# Assuming error_indices is your list of indexes to fix
# Using tqdm(error_indices) will show the progress bar in your notebook
for idx in tqdm(error_indices, desc="Correcting SQL Hallucinations"):
    
    # 1. Execute with the agentic retry logic
    # This calls build_agentic_prompt which uses the 'The' anchor internally
    # It also handles the extract_sql_from_reasoning fix for logic errors
    result, final_sql = execute_sql_with_retry(
        max_try=3, 
        index=idx, 
        db_dir=KAGGLE_DB_DIR, 
        dataset=validation_df
    )
    
    # 2. Update the 'new_pred_sql' column with the corrected SQL
    validation_df.at[idx, 'new_pred_sql'] = final_sql
    
    # 3. Optional: Logging the specific index if it fails all 3 tries
    if result is None:
        tqdm.write(f"⚠️ Index {idx} still failing after max retries.")

Correcting SQL Hallucinations:   0%|          | 0/197 [00:00<?, ?it/s]

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Attempt 1 failed on pets_1: no such table: HAS_PETS


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Attempt 2 failed on pets_1: no such table: students


...

Attempt 1 failed on dog_kennels: no such column: tt.description


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Attempt 2 failed on dog_kennels: no such column: tt.description


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Attempt 1 failed on real_estate_properties: no such column: T2.property_type_description


In [63]:
def execute_ground_truth(index, db_dir):

    sql = validation_df['query'].iloc[index]
    db_id = validation_df['db_id'].iloc[index]

    # print(f'ground truth is : {sql}')
    
    """Execute SQL against SQLite DB, return result set or None on error."""
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    # CRITICAL: Check if the file actually exists before connecting
    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None
    try:
        # 'mode=ro' tells SQLite to open it in Read-Only mode
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    # except Exception as e:

    #     return None   # invalid SQL
    except sqlite3.Error as e:
        # This will return the specific SQLite error message (e.g., "no such column: name")
        print(f"SQL Error on {db_id}: {e}")
        return str(e)
    except Exception as e:
        # This catches other issues (like file path errors)
        print(f"General Error: {e}")
        return None

result = execute_ground_truth(index = index, db_dir = KAGGLE_DB_DIR )
print(f'ground_truth_rows is : {result}')
print(len(result))

ground_truth_rows is : [('toyota',), ('plymouth',), ('amc',), ('ford',), ('datsun',), ('volkswagen',), ('peugeot',), ('audi',), ('saab',), ('bmw',), ('amc',), ('datsun',), ('chevrolet',), ('toyota',), ('ford',), ('volkswagen',), ('amc',), ('amc',), ('chevrolet',), ('mercury',), ('opel',), ('peugeot',), ('fiat',), ('toyota',), ('datsun',), ('volkswagen',), ('plymouth',), ('toyota',), ('dodge',), ('volkswagen',), ('chevrolet',), ('ford',), ('mazda',), ('volvo',), ('volkswagen',), ('peugeot',), ('renault',), ('ford',), ('datsun',), ('toyota',), ('dodge',), ('toyota',), ('amc',), ('plymouth',), ('volkswagen',), ('amc',), ('toyota',), ('chevrolet',), ('datsun',), ('mazda',), ('ford',), ('mercury',), ('fiat',), ('fiat',), ('opel',), ('audi',), ('volvo',), ('saab',), ('toyota',), ('ford',), ('amc',), ('datsun',), ('ford',), ('toyota',), ('chevrolet',), ('audi',), ('volkswagen',), ('opel',), ('toyota',), ('datsun',), ('dodge',), ('fiat',), ('fiat',), ('honda',), ('subaru',), ('fiat',), ('toyot

In [64]:
import os
import sqlite3

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
def execute_sql(sql, db_id, db_dir = KAGGLE_DB_DIR ):
    
    """Execute SQL against SQLite DB, return result set or None on error."""
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

    # CRITICAL: Check if the file actually exists before connecting
    if not os.path.exists(db_path):
        print(f"Warning: Database file not found at {db_path}")
        return None
    try:
        # 'mode=ro' tells SQLite to open it in Read-Only mode
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    # except Exception as e:
    #     return None   # invalid SQL
    except sqlite3.Error as e:
        # This will return the specific SQLite error message (e.g., "no such column: name")
        print(f"SQL Error on {db_id}: {e}")
        return str(e)
    except Exception as e:
        # This catches other issues (like file path errors)
        print(f"General Error: {e}")
        return None
    
sql = 'SELECT T1.Continent,  count(*),  T2.ContinentId FROM continents AS T1 JOIN countries AS T2 ON T1.ContId  =  T2.Continent GROUP BY T2.ContinentId'
result = execute_sql(sql, db_id = 'car_1')
print(result)

SQL Error on car_1: no such column: T2.ContinentId
no such column: T2.ContinentId


In [65]:
import os
import sqlite3
import pandas as pd

def normalize_df(df):
    """Sort columns alphabetically and rows by all columns for order-invariant comparison."""
    df = df.copy()
    # Normalize column names to lowercase
    df.columns = [c.lower().strip() for c in df.columns]
    # Sort columns alphabetically
    df = df.reindex(sorted(df.columns), axis=1)
    # Sort rows by all columns (converts to str to handle mixed types)
    df = df.apply(lambda col: col.astype(str).str.strip().str.lower())
    df = df.sort_values(by=list(df.columns)).reset_index(drop=True)
    return df

def execution_match(pred_df, gold_df):
    """Returns True if pred and gold have same content regardless of column/row order."""
    try:
        pred_norm = normalize_df(pred_df)
        gold_norm = normalize_df(gold_df)

        # print(pred_norm)
        # print(gold_norm)

        pred_norm.columns = range(len(pred_norm.columns))
        gold_norm.columns = range(len(gold_norm.columns))
        
        if pred_norm.equals(gold_norm):
            return 'correct'
    except Exception:
        return 'wrong'



def evaluate_pair(db_path, pred_sql, gold_sql):
    conn = sqlite3.connect(db_path)
    try:
        pred_df = pd.read_sql_query(pred_sql, conn)
        gold_df = pd.read_sql_query(gold_sql, conn)
        return execution_match(pred_df, gold_df)
    except Exception as e:
        return 'error'  # SQL error = wrong
    finally:
        conn.close()

# pred_sql = 'SELECT T1.Continent,  count(*),  T1.ContId FROM continents AS T1 JOIN countries AS T2 ON T1.ContId  =  T2.Continent GROUP BY T1.ContId'
# gold_sql = 'SELECT T1.ContId ,  T1.Continent ,  count(*) FROM CONTINENTS AS T1 JOIN COUNTRIES AS T2 ON T1.ContId  =  T2.Continent GROUP BY T1.ContId'

KAGGLE_DB_DIR = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/database'
db_dir = KAGGLE_DB_DIR

i = 0

pred_sql = validation_df['new_pred_sql'].iloc[i]
gold_sql = validation_df['query'].iloc[i]
db_id = validation_df['db_id'].iloc[i]

db_path = db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")

results = evaluate_pair(db_path, pred_sql, gold_sql)
results

'correct'

In [69]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
import threading
from tqdm.auto import tqdm

def execution_match(pred_df, gold_df):
    pred_norm = normalize_df(pred_df)
    gold_norm = normalize_df(gold_df)

    
    pred_norm.columns = range(len(pred_norm.columns))
    gold_norm.columns = range(len(gold_norm.columns))
    
    return 'correct' if pred_norm.equals(gold_norm) else 'wrong'

def evaluate_pair(db_path, pred_sql, gold_sql):
    conn = sqlite3.connect(db_path, check_same_thread=False, timeout=10)
    try:
        pred_df = pd.read_sql_query(pred_sql, conn)
        gold_df = pd.read_sql_query(gold_sql, conn)
        return execution_match(pred_df, gold_df)
    except Exception:
        return 'error'
    finally:
        conn.close()

def evaluate_single(args):
    i, pred_sql, gold_sql, db_id = args
    db_path = os.path.join(db_dir, db_id, f"{db_id}.sqlite")
    status = evaluate_pair(db_path, pred_sql, gold_sql)
    return i, status

def execution_accuracy_parallel(validation_df, max_workers=4):
    total = len(validation_df)
    args_list = [
        (i,
         validation_df['new_pred_sql'].iloc[i],
         validation_df['query'].iloc[i],
         validation_df['db_id'].iloc[i])
        for i in range(total)
    ]

    results = [None] * total

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(evaluate_single, args): args[0] for args in args_list}
        for future in tqdm(as_completed(futures), total=total, desc="Evaluating"):
            i = futures[future]
            try:
                i, status = future.result()
                results[i] = status
            except Exception as e:
                results[i] = 'error'

    correct  = sum(1 for r in results if r == 'correct')
    errors   = sum(1 for r in results if r == 'error')
    wrong    = sum(1 for r in results if r == 'wrong')
    accuracy = correct / total

    print(f"Execution Accuracy : {accuracy:.4f} ({correct}/{total})")
    print(f"Wrong              : {wrong}/{total}")
    print(f"Invalid SQL errors : {errors}/{total}")
    return accuracy, results
# Step 2 — evaluation (CPU, parallelized, no GPU needed)
accuracy, results = execution_accuracy_parallel(validation_df, max_workers=4)

Evaluating:   0%|          | 0/1034 [00:00<?, ?it/s]

Execution Accuracy : 0.8559 (885/1034)
Wrong              : 126/1034
Invalid SQL errors : 23/1034


In [70]:
accuracy

0.8558994197292069

In [71]:
validation_df['results'] = results
validation_df.to_csv("/kaggle/working/CoT_agent_validation_predictions.csv", index=False)
validation_df['results'].value_counts()

results
correct    885
wrong      126
error       23
Name: count, dtype: int64